# ЛР 03.2 — GridSearchCV и финальный выбор модели (Solution)

## Как проходить этот ноутбук
### Цель
- честно подобрать гиперпараметры только на `train`;
- сравнить лучшие tuned-конфигурации на `validation`;
- один раз проверить `baseline_default` и `tuned_best` на `test`.

### Входные артефакты
- `outputs/model_feature_set_decisions.csv` из notebook 1;
- те же датасеты `medical` и `finance`;
- helper-функции из `lab_utils.py`.

### Выходные артефакты
- `outputs/gridsearch_results_top.csv`
- `outputs/baseline_vs_tuned_test_results.csv`

### Где находятся обязательные самостоятельные задания
- сопоставляйте этот подробный разбор с одноименными шагами и TODO-блоками в студенческом ноутбуке;
- ориентируйтесь на markdown-блоки `Как интерпретировать результат` и `Проверь себя`.

### Что потом переносится в отчет
- шаги 1-3: раздел `5`;
- шаг 4: разделы `6` и `7`.

Это подробная версия разбора: используйте ее как образец reasoning, а не как готовый текст отчета.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util
import json

import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '03-overfitting-validation-and-hyperparameter-tuning',
    cwd.parent / '03-overfitting-validation-and-hyperparameter-tuning',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError(
        'Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 03 или из корня репозитория.'
    )

spec = importlib.util.spec_from_file_location('lab03_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

## Шаг 1. Подготовка контекста для честного тюнинга

### Что делаем
- заново воспроизводим split `train/validation/test`;
- читаем `model_feature_set_decisions.csv` как явный входной контракт;
- готовим контекст для каждой пары `dataset + model`.

### Почему это важно
- тюнинг должен стартовать не с произвольного состояния ноутбука, а с явного артефакта из notebook 1;
- если контракт между ноутбуками поврежден, это надо поймать до запуска `GridSearchCV`;
- студент должен понимать, какой именно набор признаков (feature set) сейчас идет в тюнинг.

### Что уже готово на входе
- `model_feature_set_decisions.csv` из notebook 1;
- helper `load_model_feature_set_decisions`, который валидирует CSV;
- helper `get_model_feature_set_decision`, который возвращает одну строку для пары `dataset + model`.

### Что должно получиться на выходе
- валидированная таблица `model_feature_set_decisions`;
- таблица `chosen_feature_sets`;
- словарь `split_context`, который дальше использует тюнинг и финальный финальная проверка на тестовой выборке.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_feature_set_decisions = lab.load_model_feature_set_decisions(
    expected_datasets=sorted(datasets),
    expected_models=sorted(lab.make_tuning_models()),
)

model_feature_set_decisions

,dataset,model,selected_feature_set,train_f1,validation_f1,f1_gap,abs_f1_gap,tie_break_reason
0,finance,LogisticRegression,set_B_tree,0.584270,0.586387,-0.002118,0.002118,best validation_f1
1,finance,RandomForest,set_A_wrapper,1.000000,0.500000,0.500000,0.500000,best validation_f1
2,medical,LogisticRegression,full,0.543807,0.568807,-0.025001,0.025001,best validation_f1
3,medical,RandomForest,set_C_hybrid,1.000000,0.400000,0.600000,0.600000,best validation_f1


In [3]:
# Что делаем: Обучаем модель и, при необходимости, подбираем параметры.
# Зачем: На этом шаге формируется качество модели, которое дальше анализируется в метриках и графиках.
# Как читать результат: Смотрите на итоговые метрики и выбранные параметры: они должны соответствовать ожиданиям шага.
# Типичные ошибки: Частая ошибка — случайно обучить модель на неправильном split и получить смещенную оценку качества.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
split_context = {}
selection_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_valid, x_test, y_train, y_valid, y_test = lab.train_valid_test_split_stratified(x, y)

    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name in lab.make_tuning_models().keys():
        decision_row = lab.get_model_feature_set_decision(
            decisions=model_feature_set_decisions,
            dataset_name=dataset_name,
            model_name=model_name,
        )
        feature_set_name = decision_row['selected_feature_set']
        selected_features = lab.get_feature_set_features(feature_sets, dataset_name, feature_set_name)
        # Обучаем модель на подготовленных данных.
        selector = lab.PreprocessedFeatureSelector(selected_features=selected_features).fit(x_train, y_train)
        selected_feature_names = selector.get_feature_names_out().tolist()

        split_context[(dataset_name, model_name)] = {
            'x_train': x_train,
            'x_valid': x_valid,
            'x_test': x_test,
            'y_train': y_train,
            'y_valid': y_valid,
            'y_test': y_test,
            'feature_set_name': feature_set_name,
            'selected_features': selected_features,
        }

        selection_rows.append(
            {
                'dataset': dataset_name,
                'model': model_name,
                'feature_set': feature_set_name,
                'n_train': len(x_train),
                'n_validation': len(x_valid),
                'n_test': len(x_test),
                'n_selected_features': len(selected_feature_names),
                'selected_preview': ', '.join(selected_feature_names[:5]),
            }
        )

chosen_feature_sets = pd.DataFrame(selection_rows).sort_values(['dataset', 'model']).reset_index(drop=True)
chosen_feature_sets

,dataset,model,feature_set,n_train,n_validation,n_test,n_selected_features,selected_preview
0,finance,LogisticRegression,set_B_tree,660,220,220,10,"num__loan_to_income, num__annual_income, num__credit_score, num__utilization_ratio, num__loan_amount"
1,finance,RandomForest,set_A_wrapper,660,220,220,10,"num__annual_income, num__credit_score, num__loan_to_income, num__delinquency_count, cat__previous_default_no"
2,medical,LogisticRegression,full,540,180,180,17,"num__age, num__bmi, num__systolic_bp, num__diastolic_bp, num__cholesterol"
3,medical,RandomForest,set_C_hybrid,540,180,180,9,"num__age, num__cholesterol, num__systolic_bp, num__physical_activity_hours, cat__smoking_status_never"


### Как интерпретировать результат
- `model_feature_set_decisions` — это явный входной контракт из notebook 1, а `chosen_feature_sets` показывает, как он превращается в контекст для честного тюнинга.
- Здесь важно не просто прочитать CSV, а убедиться, что у каждой пары `dataset + model` есть ровно один выбранный набор признаков (feature set).
- Таблица preview полезна для sanity-check: видно размеры split и пример transformed features.

### Пример вывода по шагу
- Notebook 2 стартует не с «магического выбора признаков», а с явного CSV-контракта, полученного на предыдущем этапе.
- Helper `load_model_feature_set_decisions` полезен именно тем, что валидирует этот контракт до запуска тяжелого тюнинга.
- Если бы здесь не было отдельного артефакта, workflow снова стал бы скрыто зависеть от прошлого ноутбука.

### Проверь себя
- В `model_feature_set_decisions` есть по одной строке на каждую пару `dataset + model`.
- `chosen_feature_sets` показывает оба датасета и обе модели.
- Вы понимаете, откуда берется `selected_features` для каждого pipeline.
- Если CSV из notebook 1 был бы поврежден, вы знаете, почему запуск должен упасть до тюнинга.

## Шаг 2. GridSearchCV только на `train`

### Что делаем
- для каждой пары `dataset + model + selected_feature_set` запускаем `GridSearchCV`;
- используем `Pipeline`, `StratifiedKFold` и multi-metric scoring;
- сохраняем top-5 конфигураций и внешний validation-summary.

### Почему это важно
- тюнинг и итоговая проверка — это разные этапы;
- preprocessing должен обучаться внутри CV, а не заранее на всем наборе;
- высокий mean CV score сам по себе еще не делает модель финальным winner.

### Что уже готово на входе
- `split_context` из шага 1;
- helper `build_model_pipeline`, который помещает preprocessing внутрь fit;
- сетки параметров из `lab.make_param_grids()`.

### Что должно получиться на выходе
- таблица `gridsearch_results_top` с top-5 строками для каждой пары `dataset + model`;
- таблица `validation_summary` с лучшей tuned-конфигурацией каждой модели на внешнем validation.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [4]:
# Что делаем: Обучаем модель и, при необходимости, подбираем параметры.
# Зачем: На этом шаге формируется качество модели, которое дальше анализируется в метриках и графиках.
# Как читать результат: Смотрите на итоговые метрики и выбранные параметры: они должны соответствовать ожиданиям шага.
# Типичные ошибки: Частая ошибка — случайно обучить модель на неправильном split и получить смещенную оценку качества.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
param_grids = lab.make_param_grids()

grid_frames = []
validation_rows = []
grid_cache = {}

# Итерируемся по объектам и последовательно накапливаем результаты.
for (dataset_name, model_name), ctx in split_context.items():
    model = clone(lab.make_tuning_models()[model_name])
    pipeline = lab.build_model_pipeline(
        model=model,
        selected_features=ctx['selected_features'],
    )
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring={
            'f1': 'f1',
            'roc_auc': 'roc_auc',
            'accuracy': 'accuracy',
        },
        refit='f1',
        cv=cv,
        n_jobs=1,
        return_train_score=False,
    )
    # Обучаем модель на подготовленных данных.
    grid.fit(ctx['x_train'], ctx['y_train'])
    grid_cache[(dataset_name, model_name)] = grid

    cv_results = pd.DataFrame(grid.cv_results_)
    grid_frames.append(
        lab.top_gridsearch_rows(
            cv_results=cv_results,
            dataset_name=dataset_name,
            feature_set_name=ctx['feature_set_name'],
            model_name=model_name,
            top_n=5,
        )
    )

    validation_metrics = lab.evaluate_fitted_model(
        grid.best_estimator_,
        ctx['x_valid'],
        ctx['y_valid'],
    )
    validation_rows.append(
        {
            'dataset': dataset_name,
            'feature_set': ctx['feature_set_name'],
            'model': model_name,
            'validation_accuracy': validation_metrics['accuracy'],
            'validation_f1': validation_metrics['f1'],
            'validation_roc_auc': validation_metrics['roc_auc'],
            'best_params_json': json.dumps(grid.best_params_, ensure_ascii=False, sort_keys=True),
        }
    )

gridsearch_results_top = pd.concat(grid_frames, ignore_index=True)
gridsearch_results_top = gridsearch_results_top.sort_values(
    ['dataset', 'model', 'rank']
).reset_index(drop=True)

validation_summary = pd.DataFrame(validation_rows).sort_values(
    ['dataset', 'validation_f1', 'validation_roc_auc', 'model'],
    ascending=[True, False, False, True],
).reset_index(drop=True)

display(gridsearch_results_top)
validation_summary

,dataset,feature_set,model,rank,params_json,mean_cv_f1,std_cv_f1,mean_cv_roc_auc,mean_cv_accuracy,mean_fit_time_sec
0,finance,set_B_tree,LogisticRegression,1,"{""model__C"": 1.0, ""model__class_weight"": ""balanced""}",0.581885,0.025393,0.679413,0.665152,0.008170
1,finance,set_B_tree,LogisticRegression,2,"{""model__C"": 10.0, ""model__class_weight"": ""balanced""}",0.579143,0.023767,0.679464,0.663636,0.008158
2,finance,set_B_tree,LogisticRegression,3,"{""model__C"": 0.1, ""model__class_weight"": ""balanced""}",0.564305,0.029352,0.679257,0.654545,0.008171
3,finance,set_B_tree,LogisticRegression,4,"{""model__C"": 0.01, ""model__class_weight"": ""balanced""}",0.543946,0.004001,0.673839,0.639394,0.007797
4,finance,set_B_tree,LogisticRegression,5,"{""model__C"": 1.0, ""model__class_weight"": null}",0.419578,0.062237,0.678828,0.657576,0.008405
5,finance,set_A_wrapper,RandomForest,1,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": 4, ""model__min_samples_leaf"": 10}",0.528601,0.032023,0.669657,0.650000,0.746270
6,finance,set_A_wrapper,RandomForest,2,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": 8, ""model__min_samples_leaf"": 10}",0.526682,0.032616,0.665346,0.653030,0.833637
7,finance,set_A_wrapper,RandomForest,3,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": null, ""model__min_samples_leaf"": 10}",0.520597,0.032270,0.665446,0.643939,0.749322
8,finance,set_A_wrapper,RandomForest,4,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": 4, ""model__min_samples_leaf"": 5}",0.519940,0.034455,0.669718,0.648485,0.761151
9,finance,set_A_wrapper,RandomForest,5,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": 4, ""model__min_samples_leaf"": 1}",0.518638,0.029289,0.669526,0.651515,0.729708


,dataset,feature_set,model,validation_accuracy,validation_f1,validation_roc_auc,best_params_json
0,finance,set_B_tree,LogisticRegression,0.640909,0.586387,0.722517,"{""model__C"": 1.0, ""model__class_weight"": ""balanced""}"
1,finance,set_A_wrapper,RandomForest,0.672727,0.560976,0.692382,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": 4, ""model__min_samples_leaf"": 10}"
2,medical,set_C_hybrid,RandomForest,0.755556,0.521739,0.785052,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": 4, ""model__min_samples_leaf"": 5}"
3,medical,full,LogisticRegression,0.705556,0.513761,0.838516,"{""model__C"": 0.01, ""model__class_weight"": ""balanced""}"


### Как интерпретировать результат
- `gridsearch_results_top` отвечает на вопрос, какие конфигурации лучше всего выглядели внутри CV на train-части.
- `validation_summary` отвечает на следующий вопрос: как лучший tuned-кандидат каждой модели ведет себя уже на отдельном validation.
- Именно поэтому тут две таблицы: CV-результаты и внешний validation-check нельзя смешивать.

### Пример вывода по шагу
- Высокий `mean_cv_f1` еще не делает модель итоговым winner: после CV она должна пережить честную проверку на validation.
- `Pipeline` нужен не как формальность, а чтобы preprocessing обучался внутри фолдов, а не заранее на всем train.
- Для отчета здесь важно уметь объяснить не только параметры, но и сам workflow: сначала CV на train, потом сравнение на validation.

### Проверь себя
- `GridSearchCV` работал только на `train`.
- В `gridsearch_results_top` для каждой пары `dataset + model` есть 5 строк.
- В `validation_summary` у каждой модели есть `validation_f1` и `best_params_json`.
- Вы можете словами объяснить, зачем нужны и CV-таблица, и внешний validation-check.

## Шаг 3. Выбор итоговой tuned-модели по validation

### Что делаем
- сравниваем лучшие tuned-кандидаты между моделями внутри каждого dataset;
- применяем явное правило выбора winner;
- фиксируем dataset-level решение в `final_choice_summary`.

### Почему это важно
- до этого момента мы выбирали набор признаков (feature set) и параметры, но еще не выбирали финальную model family;
- итоговый winner должен определяться validation-качеством, а не впечатлением от сложности модели;
- этот шаг завершает этап отбора перед единственным обращением к `test`.

### Что уже готово на входе
- `validation_summary` из шага 2;
- helper `choose_validation_winner`, который реализует правило `max validation_f1 -> max validation_roc_auc -> prefer LogisticRegression`.

### Что должно получиться на выходе
- таблица `final_choice_summary` с двумя строками: по одному winner на `medical` и `finance`.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [5]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
winner_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name in sorted(datasets):
    winner = lab.choose_validation_winner(validation_summary, dataset_name)
    winner_rows.append(
        {
            'dataset': dataset_name,
            'model': winner['model'],
            'feature_set': winner['feature_set'],
            'validation_f1': winner['validation_f1'],
            'validation_roc_auc': winner['validation_roc_auc'],
            'best_params_json': winner['best_params_json'],
            'decision_rule': 'max validation_f1 -> max validation_roc_auc -> prefer LogisticRegression',
        }
    )

final_choice_summary = pd.DataFrame(winner_rows).sort_values('dataset').reset_index(drop=True)
final_choice_summary

,dataset,model,feature_set,validation_f1,validation_roc_auc,best_params_json,decision_rule
0,finance,LogisticRegression,set_B_tree,0.586387,0.722517,"{""model__C"": 1.0, ""model__class_weight"": ""balanced""}",max validation_f1 -> max validation_roc_auc -> prefer LogisticRegression
1,medical,RandomForest,set_C_hybrid,0.521739,0.785052,"{""model__class_weight"": ""balanced_subsample"", ""model__max_depth"": 4, ""model__min_samples_leaf"": 5}",max validation_f1 -> max validation_roc_auc -> prefer LogisticRegression


### Как интерпретировать результат
- `final_choice_summary` — это уже dataset-level решение: какая tuned-модель стала финальным кандидатом после сравнения на validation.
- Helper `choose_validation_winner` делает правило выбора прозрачным: сначала `validation_f1`, потом `validation_roc_auc`, затем предпочтение `LogisticRegression`.
- Нормальный sanity-check: в таблице должно быть ровно 2 строки — по одной на каждый dataset.

### Пример вывода по шагу
- Здесь важно не перепутать два уровня выбора: сначала набор признаков (feature set) выбирался отдельно для каждой модели, а теперь мы сравниваем уже лучшие tuned-кандидаты между моделями.
- Даже если одна модель выглядит сложнее и «умнее», победа все равно определяется validation-метрикой, а не впечатлением от алгоритма.
- Для отчета отсюда удобно брать итоговое объяснение: почему именно эта tuned-пара пошла в финальный финальная проверка на тестовой выборке.

### Проверь себя
- В `final_choice_summary` ровно 2 строки.
- Для каждого dataset указаны `model`, `feature_set` и `best_params_json`.
- Вы можете объяснить, почему этот шаг идет по validation, а не по test.
- После этого шага этап отбора завершен, и можно переходить к единственному финальная проверка на тестовой выборке.

## Шаг 4. Один финальный финальная проверка на тестовой выборке

### Что делаем
- берем только winning `model + feature_set` для каждого dataset;
- обучаем `baseline_default` и `tuned_best` на `train + validation`;
- сравниваем их на `test` ровно один раз.

### Почему это важно
- это честная финальная проверка, а не еще один этап выбора;
- обе variant сравниваются в одинаковых условиях;
- именно здесь видно, помог ли тюнинг на настоящем holdout.

### Что уже готово на входе
- `final_choice_summary`;
- `split_context`;
- helper `fit_and_evaluate_pipeline` для единого измерения метрик и времени fit.

### Что должно получиться на выходе
- таблица `baseline_vs_tuned_test_results` с двумя variant-ами на каждый dataset.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [6]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
comparison_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, winner in final_choice_summary.set_index('dataset').iterrows():
    model_name = winner['model']
    ctx = split_context[(dataset_name, model_name)]

    x_train_valid = pd.concat([ctx['x_train'], ctx['x_valid']], axis=0).reset_index(drop=True)
    y_train_valid = pd.concat([ctx['y_train'], ctx['y_valid']], axis=0).reset_index(drop=True)

    baseline_pipeline = lab.build_model_pipeline(
        model=clone(lab.make_default_models()[model_name]),
        selected_features=ctx['selected_features'],
    )
    tuned_pipeline = lab.build_model_pipeline(
        model=clone(lab.make_tuning_models()[model_name]),
        selected_features=ctx['selected_features'],
    )
    tuned_pipeline.set_params(**json.loads(winner['best_params_json']))

    _, baseline_metrics = lab.fit_and_evaluate_pipeline(
        estimator=baseline_pipeline,
        x_train=x_train_valid,
        y_train=y_train_valid,
        x_eval=ctx['x_test'],
        y_eval=ctx['y_test'],
    )
    _, tuned_metrics = lab.fit_and_evaluate_pipeline(
        estimator=tuned_pipeline,
        x_train=x_train_valid,
        y_train=y_train_valid,
        x_eval=ctx['x_test'],
        y_eval=ctx['y_test'],
    )

    # Итерируемся по объектам и последовательно накапливаем результаты.
    for variant_name, metrics in [('baseline_default', baseline_metrics), ('tuned_best', tuned_metrics)]:
        comparison_rows.append(
            {
                'dataset': dataset_name,
                'feature_set': winner['feature_set'],
                'model': model_name,
                'variant': variant_name,
                'accuracy': metrics['accuracy'],
                'f1': metrics['f1'],
                'roc_auc': metrics['roc_auc'],
                'fit_time_sec': metrics['fit_time_sec'],
            }
        )

baseline_vs_tuned_test_results = pd.DataFrame(comparison_rows)
baseline_vs_tuned_test_results = baseline_vs_tuned_test_results.sort_values(
    ['dataset', 'model', 'variant']
).reset_index(drop=True)
baseline_vs_tuned_test_results

,dataset,feature_set,model,variant,accuracy,f1,roc_auc,fit_time_sec
0,finance,set_B_tree,LogisticRegression,baseline_default,0.631818,0.537143,0.694238,0.012456
1,finance,set_B_tree,LogisticRegression,tuned_best,0.631818,0.537143,0.694238,0.012153
2,medical,set_C_hybrid,RandomForest,baseline_default,0.788889,0.321429,0.728678,0.811109
3,medical,set_C_hybrid,RandomForest,tuned_best,0.700000,0.437500,0.723950,0.772665


### Как интерпретировать результат
- В этой таблице baseline и tuned сравниваются только на одной и той же winning-паре `model + feature_set` и на одном и том же `train + validation`.
- Это важно: мы не меняем условия сравнения между variant-ами и тем самым не подмешиваем лишние факторы.
- Если tuned-модель не выигрывает на test, это не значит, что workflow был плохим; это значит, что честная проверка сработала по назначению.

### Пример вывода по шагу
- Финальный финальная проверка на тестовой выборке — это проверка процедуры, а не конкурс «кто красивее выглядит в таблице».
- Возможный проигрыш tuned-модели baseline на test — методически важный результат: tuning не гарантирует рост на holdout.
- Для отчета отсюда нужно взять не только числа, но и объяснение, почему один честный финальная проверка на тестовой выборке важнее десятка подгонок по holdout.

### Проверь себя
- В таблице есть `baseline_default` и `tuned_best` для каждого dataset.
- Baseline и tuned сравниваются на одном и том же winning `model + feature_set`.
- `test` использован только один раз после завершения всех решений.
- Вы можете объяснить, почему проигрыш tuned-модели baseline не делает лабораторную «неудачной».

## Шаг 5. Экспорт обязательных артефактов

### Что делаем
- проверяем контракты итоговых таблиц;
- сохраняем 2 обязательных CSV в `outputs/`;
- фиксируем завершение ЛР 03 на уровне артефактов.

### Почему это важно
- отчет должен опираться на явные таблицы, а не на случайно оставшиеся outputs в ноутбуке;
- сохраненные CSV отделяют train/CV-stage от final holdout-stage;
- это последний технический шаг перед сдачей ЛР.

### Что уже готово на входе
- `gridsearch_results_top`;
- `baseline_vs_tuned_test_results`.

### Что должно получиться на выходе
- два CSV в `outputs/`;
- полностью закрытый обязательный маршрут ЛР 03.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [7]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
required_grid_columns = {
    'dataset',
    'feature_set',
    'model',
    'rank',
    'params_json',
    'mean_cv_f1',
    'std_cv_f1',
    'mean_cv_roc_auc',
    'mean_cv_accuracy',
    'mean_fit_time_sec',
}
required_test_columns = {
    'dataset',
    'feature_set',
    'model',
    'variant',
    'accuracy',
    'f1',
    'roc_auc',
    'fit_time_sec',
}

# Проверяем обязательное условие корректности шага.
assert set(gridsearch_results_top.columns) == required_grid_columns
assert set(baseline_vs_tuned_test_results.columns) == required_test_columns

# Сохраняем таблицу артефакта в CSV.
gridsearch_results_top.to_csv(OUTPUT_DIR / 'gridsearch_results_top.csv', index=False)
baseline_vs_tuned_test_results.to_csv(OUTPUT_DIR / 'baseline_vs_tuned_test_results.csv', index=False)

print('Сохранены outputs/gridsearch_results_top.csv и outputs/baseline_vs_tuned_test_results.csv')

Сохранены outputs/gridsearch_results_top.csv и outputs/baseline_vs_tuned_test_results.csv


### Как интерпретировать результат
- Экспорт завершает ЛР 03 как полный честный pipeline: train/CV-stage и final holdout-stage теперь сохранены отдельно и прозрачно.
- Именно эти два CSV делают итоговую историю эксперимента воспроизводимой и понятной без повторного запуска ячеек.
- После этого шага уже можно собирать разделы `5`, `6` и `7` отчета и формулировать практическую рекомендацию.

### Пример вывода по шагу
- `gridsearch_results_top.csv` отвечает за тюнинг и сравнение внутри train/CV-stage.
- `baseline_vs_tuned_test_results.csv` отвечает за единственный честный ответ на вопрос: помог ли тюнинг на holdout.
- Такое разделение важно методически: студент видит, где был выбор, а где была только финальная проверка.


### Проверь себя
- В `outputs/` лежат оба финальных CSV.
- Вы можете восстановить ход решения без обращения к скрытому состоянию ноутбука.
- Все разделы отчета по notebook 2 уже обеспечены данными.
- Вы можете отдельно объяснить и CV-stage, и final test-stage.
